# <b><span style='color:#F1A424'>|</span> CS506: <span style='color:#F1A424'>Thyroid Cancer Prediction Project</span><span style='color:#ABABAB'> [Advance]</span></b> 

***

### <b><span style='color:#F1A424'>Table of Contents</span></b> <a class='anchor' id='top'></a>
<div style=" background-color:#3b3745; padding: 13px 13px; border-radius: 8px; color: white">
    <li> <a href="#install_libraries">Install libraries</a></li>
    <li> <a href="#introduction">Introduction</a></li>
    <li><a href="#import_libraries">Import Libraries</a></li>
    <li><a href="#configuration">Configuration</a></li>
    <li><a href="#utils">Utils</a></li>
    <li><a href="#load_data">Load Data</a></li>
    <li><a href="#preprocessing">Data Pre-processing</a></li>
    <li><a href="#validation">Validation</a></li>
    <li><a href="#modeling">Modeling</a></li>
</div>


# <b><span style='color:#F1A424'>|</span> Introduction</b><a class='anchor' id='introduction'></a> [↑](#top) 

***



# <b><span style='color:#F1A424'>|</span> Install Libraries</b><a class='anchor' id='install_libraries'></a> [↑](#top) 

***

Install all the required libraries for this notebook.

# <b><span style='color:#F1A424'>|</span> Import Libraries</b><a class='anchor' id='import_libraries'></a> [↑](#top) 

***

Import all the required libraries for this notebook.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle
import plotly.express as px
import sklearn
from sklearn.experimental import enable_iterative_imputer  
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler


from glob import glob

from sklearn.preprocessing import LabelEncoder

# <b><span style='color:#F1A424'>|</span> Configuration</b><a class='anchor' id='configuration'></a> [↑](#top) 

***

Configuration for this notebook.

In [2]:
class paths:
    OUTPUT = "/kaggle/working"
    TRAIN_CSV = "/kaggle/input/private-dataset/allhypo.data"
    TEST_CSV = "/kaggle/input/private-dataset/allhypo.test"

# <b><span style='color:#F1A424'>|</span> Utils</b><a class='anchor' id='utils'></a> [↑](#top) 

***

Utils for this notebook.

In [3]:
def sep():
    print("_"*100)

# <b><span style='color:#F1A424'>|</span> Load Data</b><a class='anchor' id='load_data'></a> [↑](#top) 

***

Load data for this notebook.

In [4]:
COLS_ALL = [
    "age", "sex", "on_thyroxine", "query_on_thyroxine", "on_antithyroid_medication",
    "sick", "pregnant", "thyroid_surgery", "I131_treatment", "query_hypothyroid",
    "query_hyperthyroid", "lithium", "goitre", "tumor", "hypopituitary", "psych",
    "TSH_measured", "TSH", "T3_measured", "T3", "TT4_measured", "TT4", "T4U_measured",
    "T4U", "FTI_measured", "FTI", "TBG_measured", "TBG", "referral_source",
    "target"
]
COLS_CATEGORICAL = [
    "sex", "on_thyroxine", "query_on_thyroxine", "on_antithyroid_medication", "sick",                          
    "pregnant", "thyroid_surgery", "I131_treatment", "query_hypothyroid", "query_hyperthyroid",            
    "lithium", "goitre", "tumor", "hypopituitary", "psych", "TSH_measured", "T3_measured",                   
    "TT4_measured", "T4U_measured", "FTI_measured", "TBG_measured", "referral_source"               
]
COLS_NUMERICAL = [
    "age","TSH","T3","TT4","T4U","FTI","TBG"
]
COLS_TARGET = "target"

df_train = pd.read_csv(
    paths.TRAIN_CSV,
    header=None,
    names=COLS_ALL,
    na_values=["?"]
)

df_test = pd.read_csv(
    paths.TEST_CSV,
    header=None,
    names=COLS_ALL,
    na_values=["?"]
)

print(df_train.shape)
print(df_test.shape)

display(df_train.head())
display(df_test.head())

(2800, 30)
(972, 30)


,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,...,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,TBG_measured,TBG,referral_source,target
0,41.0,F,f,f,f,f,f,f,f,f,...,t,125.0,t,1.14,t,109.0,f,NaN,SVHC,negative.|3733
1,23.0,F,f,f,f,f,f,f,f,f,...,t,102.0,f,NaN,f,NaN,f,NaN,other,negative.|1442
2,46.0,M,f,f,f,f,f,f,f,f,...,t,109.0,t,0.91,t,120.0,f,NaN,other,negative.|2965
3,70.0,F,t,f,f,f,f,f,f,f,...,t,175.0,f,NaN,f,NaN,f,NaN,other,negative.|806
4,70.0,F,f,f,f,f,f,f,f,f,...,t,61.0,t,0.87,t,70.0,f,NaN,SVI,negative.|2807


,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,...,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,TBG_measured,TBG,referral_source,target
0,35,F,f,f,f,f,f,f,f,f,...,f,NaN,f,NaN,f,NaN,f,NaN,other,negative.|219
1,63,M,f,f,f,f,f,f,f,f,...,t,108.0,t,0.96,t,113.0,f,NaN,SVI,negative.|2059
2,25,F,f,f,f,f,f,f,f,f,...,t,61.0,t,0.82,t,75.0,f,NaN,SVHD,negative.|399
3,53,F,f,f,f,f,f,f,f,t,...,t,145.0,t,1.03,t,141.0,f,NaN,other,negative.|1911
4,92,F,f,f,f,f,f,f,f,f,...,t,120.0,t,0.84,t,143.0,f,NaN,SVI,negative.|487


# <b><span style='color:#F1A424'>|</span> Visualization</b><a class='anchor' id='visualization'></a> [↑](#top) 

***

Visualization.

# <b><span style='color:#F1A424'>|</span> Data Preprocessing </b><a class='anchor' id='preprocessing'></a> [↑](#top) 

***

Data pre-processing.

### <b><span style='color:#F1A424'>Label Encoding</span></b>

In [5]:
os.makedirs(paths.OUTPUT, exist_ok=True)

encoders = {}

for col in COLS_CATEGORICAL:
    encoder = LabelEncoder()

    train_values = df_train[col].fillna("Missing").astype(str)
    test_values = df_test[col].fillna("Missing").astype(str)

    encoder.fit(train_values)

    df_train[col] = encoder.transform(train_values)

    mapping = {
        category: idx
        for idx, category in enumerate(encoder.classes_)
    }

    df_test[col] = test_values.map(mapping).fillna(-1).astype(int)

    encoders[col] = encoder

# Save encoders
with open(os.path.join(paths.OUTPUT, "label_encoders.pkl"), "wb") as f:
    pickle.dump(encoders, f)

display(df_train.head())
display(df_test.head())

,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,...,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,TBG_measured,TBG,referral_source,target
0,41.0,0,0,0,0,0,0,0,0,0,...,1,125.0,1,1.14,1,109.0,0,NaN,1,negative.|3733
1,23.0,0,0,0,0,0,0,0,0,0,...,1,102.0,0,NaN,0,NaN,0,NaN,4,negative.|1442
2,46.0,1,0,0,0,0,0,0,0,0,...,1,109.0,1,0.91,1,120.0,0,NaN,4,negative.|2965
3,70.0,0,1,0,0,0,0,0,0,0,...,1,175.0,0,NaN,0,NaN,0,NaN,4,negative.|806
4,70.0,0,0,0,0,0,0,0,0,0,...,1,61.0,1,0.87,1,70.0,0,NaN,3,negative.|2807


,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,...,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,TBG_measured,TBG,referral_source,target
0,35,0,0,0,0,0,0,0,0,0,...,0,NaN,0,NaN,0,NaN,0,NaN,4,negative.|219
1,63,1,0,0,0,0,0,0,0,0,...,1,108.0,1,0.96,1,113.0,0,NaN,3,negative.|2059
2,25,0,0,0,0,0,0,0,0,0,...,1,61.0,1,0.82,1,75.0,0,NaN,2,negative.|399
3,53,0,0,0,0,0,0,0,0,1,...,1,145.0,1,1.03,1,141.0,0,NaN,4,negative.|1911
4,92,0,0,0,0,0,0,0,0,0,...,1,120.0,1,0.84,1,143.0,0,NaN,3,negative.|487


### <b><span style='color:#F1A424'>Cap outliers using IQR</span></b>

In [6]:
def cap_outliers_iqr(df_train, df_test, columns, factor=1.5):
    bounds = {}

    for col in columns:
        q1 = df_train[col].quantile(0.25)
        q3 = df_train[col].quantile(0.75)

        iqr = q3 - q1

        lower = q1 - factor * iqr
        upper = q3 + factor * iqr

        df_train[col] = df_train[col].clip(lower, upper)
        df_test[col] = df_test[col].clip(lower, upper)

        bounds[col] = (lower, upper)

    return df_train, df_test, bounds


df_train, df_test, iqr_bounds = cap_outliers_iqr(df_train, df_test, COLS_NUMERICAL)
display(df_train.head())
display(df_test.head())

,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,...,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,TBG_measured,TBG,referral_source,target
0,41.0,0,0,0,0,0,0,0,0,0,...,1,125.0,1,1.14,1,109.0,0,NaN,1,negative.|3733
1,23.0,0,0,0,0,0,0,0,0,0,...,1,102.0,0,NaN,0,NaN,0,NaN,4,negative.|1442
2,46.0,1,0,0,0,0,0,0,0,0,...,1,109.0,1,0.91,1,120.0,0,NaN,4,negative.|2965
3,70.0,0,1,0,0,0,0,0,0,0,...,1,175.0,0,NaN,0,NaN,0,NaN,4,negative.|806
4,70.0,0,0,0,0,0,0,0,0,0,...,1,61.0,1,0.87,1,70.0,0,NaN,3,negative.|2807


,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,...,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,TBG_measured,TBG,referral_source,target
0,35,0,0,0,0,0,0,0,0,0,...,0,NaN,0,NaN,0,NaN,0,NaN,4,negative.|219
1,63,1,0,0,0,0,0,0,0,0,...,1,108.0,1,0.96,1,113.0,0,NaN,3,negative.|2059
2,25,0,0,0,0,0,0,0,0,0,...,1,61.0,1,0.82,1,75.0,0,NaN,2,negative.|399
3,53,0,0,0,0,0,0,0,0,1,...,1,145.0,1,1.03,1,141.0,0,NaN,4,negative.|1911
4,92,0,0,0,0,0,0,0,0,0,...,1,120.0,1,0.84,1,143.0,0,NaN,3,negative.|487


### <b><span style='color:#F1A424'>Drop column</span></b>

In [7]:
df_train.drop(columns="TBG", inplace=True)
df_test.drop(columns="TBG", inplace=True)
display(df_train.head())
display(df_test.head())

,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,...,T3,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,TBG_measured,referral_source,target
0,41.0,0,0,0,0,0,0,0,0,0,...,2.5,1,125.0,1,1.14,1,109.0,0,1,negative.|3733
1,23.0,0,0,0,0,0,0,0,0,0,...,2.0,1,102.0,0,NaN,0,NaN,0,4,negative.|1442
2,46.0,1,0,0,0,0,0,0,0,0,...,NaN,1,109.0,1,0.91,1,120.0,0,4,negative.|2965
3,70.0,0,1,0,0,0,0,0,0,0,...,1.9,1,175.0,0,NaN,0,NaN,0,4,negative.|806
4,70.0,0,0,0,0,0,0,0,0,0,...,1.2,1,61.0,1,0.87,1,70.0,0,3,negative.|2807


,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,...,T3,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,TBG_measured,referral_source,target
0,35,0,0,0,0,0,0,0,0,0,...,NaN,0,NaN,0,NaN,0,NaN,0,4,negative.|219
1,63,1,0,0,0,0,0,0,0,0,...,2.5,1,108.0,1,0.96,1,113.0,0,3,negative.|2059
2,25,0,0,0,0,0,0,0,0,0,...,2.4,1,61.0,1,0.82,1,75.0,0,2,negative.|399
3,53,0,0,0,0,0,0,0,0,1,...,2.1,1,145.0,1,1.03,1,141.0,0,4,negative.|1911
4,92,0,0,0,0,0,0,0,0,0,...,1.3,1,120.0,1,0.84,1,143.0,0,3,negative.|487


### <b><span style='color:#F1A424'>Imputation</span></b>

In [8]:

NUM_COLS = [c for c in COLS_NUMERICAL if c != "TBG"]
FEATURES = [
    c for c in df_train.columns
    if c not in ("target","patient_id") and df_train[c].nunique() > 1
] #only keep columns that actually have unique values and aren't just constant throughout
NUM_IDX = [FEATURES.index(c) for c in NUM_COLS]

print("NaNs before (train):\n", df_train[NUM_COLS].isna().sum())

low = df_train[FEATURES].min().to_numpy()
high = df_train[FEATURES].max().to_numpy()

#Try a random forest regressor as an imputer
imputer = IterativeImputer(
    estimator = RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=5,
        n_jobs = -1,
        random_state = 42
    ),
    max_iter=10,
    min_value=low,
    max_value=high,
    random_state=42
)

X_train_imputed = imputer.fit_transform(df_train[FEATURES])
X_test_imputed = imputer.transform(df_test[FEATURES])

df_train[NUM_COLS] = X_train_imputed[:,NUM_IDX]
df_test[NUM_COLS] = X_test_imputed[:,NUM_IDX]



NaNs before (train):
 age      1
TSH    284
T3     585
TT4    184
T4U    297
FTI    295
dtype: int64


/usr/local/lib/python3.13/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


### <b><span style='color:#F1A424'>Scaling</span></b>

In [9]:
scaler = StandardScaler()
df_train[NUM_COLS] = scaler.fit_transform(df_train[NUM_COLS])
df_test[NUM_COLS] = scaler.transform(df_test[NUM_COLS])


print("NaNs after (train):", df_train[NUM_COLS].isna().sum().sum())
print("NaNs after (test): ", df_test[NUM_COLS].isna().sum().sum())
display(df_train[NUM_COLS].describe().loc[["mean", "std"]])

#Save Imputer and Scaler
with open(os.path.join(paths.OUTPUT, "imputer.pkl"), "wb") as f:
    pickle.dump(imputer, f)
with open(os.path.join(paths.OUTPUT, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

NaNs after (train): 0
NaNs after (test):  0


,age,TSH,T3,TT4,T4U,FTI
mean,-1.674851e-16,3.267228e-17,-2.131628e-16,4.732722e-16,3.273572e-16,-4.269601e-16
std,1.000179e+00,1.000179e+00,1.000179e+00,1.000179e+00,1.000179e+00,1.000179e+00


### <b><span style='color:#F1A424'>Transform target</span></b>

In [10]:
for df in [df_train, df_test]:
    df[["target", "patient_id"]] = df["target"].str.split("|", n=1, expand=True, regex=False)
    df["target"] = df["target"].str.rstrip(".")
    df["patient_id"] = pd.to_numeric(df["patient_id"])

display(df_train.head())
display(df_test.head())

,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,...,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,TBG_measured,referral_source,target,patient_id
0,-0.563917,0,0,0,0,0,0,0,0,0,...,1,0.572866,1,0.957907,1,-0.008191,0,1,negative,3733
1,-1.510411,0,0,0,0,0,0,0,0,0,...,1,-0.198590,0,0.000024,0,-0.230954,0,4,negative,1442
2,-0.301002,1,0,0,0,0,0,0,0,0,...,1,0.036201,1,-0.508348,1,0.414825,0,4,negative,2965
3,0.960991,0,1,0,0,0,0,0,0,0,...,1,2.249946,0,-0.129854,0,2.333847,0,4,negative,806
4,0.960991,0,0,0,0,0,0,0,0,0,...,1,-1.573795,1,-0.763349,1,-1.507976,0,3,negative,2807


,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,...,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,TBG_measured,referral_source,target,patient_id
0,-0.879415,0,0,0,0,0,0,0,0,0,...,0,-0.178770,0,-0.009365,0,-0.208025,0,4,negative,219
1,0.592910,1,0,0,0,0,0,0,0,0,...,1,0.002659,1,-0.189597,1,0.145633,0,3,negative,2059
2,-1.405245,0,0,0,0,0,0,0,0,0,...,1,-1.573795,1,-1.082100,1,-1.315696,0,2,negative,399
3,0.067079,0,0,0,0,0,0,0,0,1,...,1,1.243698,1,0.256655,1,1.222402,0,4,negative,1911
4,2.117817,0,0,0,0,0,0,0,0,0,...,1,0.405158,1,-0.954600,1,1.299314,0,3,negative,487


In [11]:
TARGET_MAPPING = {
    "negative": 0,
    "compensated hypothyroid": 1,
    "primary hypothyroid": 1,
    "secondary hypothyroid": 1
}
print(df_train.target.value_counts()), sep()
print(df_test.target.value_counts()), sep()

df_train["target"] = df_train["target"].map(TARGET_MAPPING)
df_test["target"] = df_test["target"].map(TARGET_MAPPING)

print(df_train.target.value_counts()), sep()
print(df_test.target.value_counts())

target
negative                   2580
compensated hypothyroid     154
primary hypothyroid          64
secondary hypothyroid         2
Name: count, dtype: int64
____________________________________________________________________________________________________
target
negative                   901
compensated hypothyroid     40
primary hypothyroid         31
Name: count, dtype: int64
____________________________________________________________________________________________________
target
0    2580
1     220
Name: count, dtype: int64
____________________________________________________________________________________________________
target
0    901
1     71
Name: count, dtype: int64


# <b><span style='color:#F1A424'>|</span> Validation</b><a class='anchor' id='validation'></a> [↑](#top) 

***

Validation.

# <b><span style='color:#F1A424'>|</span> Modeling</b><a class='anchor' id='modeling'></a> [↑](#top) 

***

Modeling.